In [1]:
from pathlib import Path

In [2]:
pwd = Path.cwd()
data_dir = pwd.parent / "hordeum_panicoid_hgt" / "positive_protein_coding" / "trees_from_orthogroups" / "confirmed_hgt"

In [3]:
import pandas as pd

# Cross-reference gene-orthogroup mapping with Eggnog results (per gene)

### We'll factor in HGT/native class

In [4]:
genes_class = pd.read_csv(data_dir / "genes_classification", names=["orthogroup", "gene", "cat"], sep='\t', header=None)
genes_class.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   orthogroup  1739 non-null   object
 1   gene        1739 non-null   object
 2   cat         1739 non-null   object
dtypes: object(3)
memory usage: 40.9+ KB


Deleting two orthogroups discarded based on trees

In [6]:
groups_to_delete = ["N0.HOG0016866", "N0.HOG0039270p2"]
genes_class = genes_class[~genes_class.orthogroup.isin(groups_to_delete)]

In [7]:
native_groups = genes_class.orthogroup[genes_class.cat == "nat"]
native_groups = native_groups.drop_duplicates()

In [8]:
genes_class["species"] = genes_class.gene.apply(lambda x: x.split('.')[0])

In [9]:
genes_class.columns

Index(['orthogroup', 'gene', 'cat', 'species'], dtype='object')

In [29]:
genes_class = genes_class.drop('species', axis=1)

In [30]:
hgt_class = genes_class[(genes_class.cat=="hgt") | (genes_class.cat=="hgt?")]
nat_class = genes_class[genes_class.cat=="nat"]
oth_class = genes_class[genes_class.cat=="oth"]

In [11]:
hgt_class.shape

(737, 4)

In [12]:
hgt = pwd.parent / "hordeum_panicoid_hgt" 
annot_dir = hgt / "annot"

In [13]:
ortho_dir = annot_dir / "orthogroups" 
eggnog_go = pd.read_csv(ortho_dir / "all_groups_eggnog_mapper.tsv", sep='\t')
eggnog_go.head()

,#query,seed_ortholog,evalue,score,eggNOG_OGs,max_annot_lvl,COG_category,Description,Preferred_name,GOs,...,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
0,HMARINUM.BCC2001.r1.2HG00091550.1:0-1269,4513.MLOC_70630.1,7.730000e-207,587.0,"COG2124@1|root,KOG0156@2759|Eukaryota,37QA7@33...",35493|Streptophyta,Q,Belongs to the cytochrome P450 family,-,-,...,ko:K00512,"ko00140,ko01100,ko04913,ko04917,ko04927,ko0493...","M00109,M00110","R02211,R03783,R04852,R04853,R08517,R08518","RC00607,RC00660,RC00923,RC01222","ko00000,ko00001,ko00002,ko00199,ko01000",-,-,-,p450
1,HMARINUM.BCC2001.r1.2HG00091580.1:0-441,37682.EMT09706,3.090000e-103,298.0,"2E0DQ@1|root,2S7UD@2759|Eukaryota,37UTH@33090|...",35493|Streptophyta,S,PLAC8 family,-,-,...,-,-,-,-,-,-,-,-,-,PLAC8
2,HMARINUM.BCC2001.r1.2HG00091590.1:0-1584,4513.MLOC_49936.1,0.000000e+00,967.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
3,HMARINUM.BCC2001.r1.2HG00091600.2:0-1593,4513.MLOC_77123.1,0.000000e+00,984.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
4,HMARINUM.BCC2001.r1.2HG00091610.2:0-663,4538.ORGLA06G0215700.1,8.710000e-73,227.0,"KOG1075@1|root,KOG1075@2759|Eukaryota,384BG@33...",35493|Streptophyta,S,Reverse transcriptase-like,-,-,...,-,-,-,-,-,-,-,-,-,RVT_3


In [14]:
eggnog_go = eggnog_go.rename(columns={'#query': 'gene', 'GOs': 'go'})

In [15]:
eggnog_go.gene = eggnog_go.gene.apply(lambda x: x.split(':')[0])

In [16]:
eggnog_go = eggnog_go.drop_duplicates()
eggnog_go.shape

(925590, 21)

In [17]:
eggnog_go = eggnog_go[eggnog_go.gene.str.startswith('H')]

In [18]:
eggnog_go_ess = eggnog_go[["gene", "go"]]
eggnog_go_ess.head()

,gene,go
0,HMARINUM.BCC2001.r1.2HG00091550.1,-
1,HMARINUM.BCC2001.r1.2HG00091580.1,-
2,HMARINUM.BCC2001.r1.2HG00091590.1,-
3,HMARINUM.BCC2001.r1.2HG00091600.2,-
4,HMARINUM.BCC2001.r1.2HG00091610.2,-


In [19]:
orthogroup_dir = hgt / "positive_protein_coding" / "orthogroup_clusters"
groups = pd.read_csv(orthogroup_dir / "gene_orthogroup", sep='\t', names=["orthogroup", "gene"])
groups.head()

,orthogroup,gene
0,N0.HOG0000000,HEUCLASTON.NGB90233.r1.7HG00797460.1
1,N0.HOG0000000,HORVU.MOREX.PROJ.5HG00772420.1
2,N0.HOG0000000,HORVU.MOREX.PROJ.3HG00275700.1
3,N0.HOG0000001,HBREVISUBULATUM.PI401390.r1.7HG01051700.1
4,N0.HOG0000001,HCALIFORNICUM.BCC2058.r1.2HG00226540.1


In [20]:
eggnog_groups = pd.merge(groups, eggnog_go_ess, on="gene", how="left")
eggnog_groups.shape

(1027577, 3)

In [21]:
eggnog_groups = eggnog_groups.fillna('-')
eggnog_groups.go = eggnog_groups.go.str.split(',')

In [31]:
eggnog_hgt = pd.merge(hgt_class, eggnog_groups, on=["orthogroup", "gene"])
eggnog_nat = pd.merge(nat_class, eggnog_groups, on=["orthogroup", "gene"])
# eggnog_oth = pd.merge(oth_class, eggnog_groups, on=["orthogroup", "gene"]) - žádné GO

In [32]:
eggnog_hgt

,orthogroup,gene,cat,go
0,N0.HOG0002063,HCALIFORNICUM.BCC2058.r1.3HG00324370.1,hgt,[-]
1,N0.HOG0002063,HCHILENSE.GRA1000.r1.6HG00734980.1,hgt,[-]
2,N0.HOG0002063,HCORDOBENSE.BCC2067.r1.6HG00711730.2,hgt,[-]
3,N0.HOG0002063,HCORDOBENSE.BCC2067.r1.1HG00069650.1,hgt,[-]
4,N0.HOG0002063,HERECTIFOLIUM.NGB6816.r1.7HG00932300.1,hgt,[-]
...,...,...,...,...
732,NUn.HOG0062451,HFLEXUOSUM.BCC2023.r1.5HG00555880.1,hgt,"[GO:0001530, GO:0003674, GO:0005488, GO:000557..."
733,NUn.HOG0066919,HMUTICUM.GB90062.r1.5HG00510610.1,hgt,[-]
734,NUn.HOG0067333,HPATAGONICUM.BCC2065.r1.6HG00816250.2,hgt,"[GO:0003674, GO:0005488, GO:0005575, GO:000562..."
735,NUn.HOG0067696,HPUBIFLORUM.BCC2028.r1.5HG00648250.1,hgt,"[GO:0000988, GO:0000990, GO:0003674, GO:000557..."


In [33]:
eggnog_nat

,orthogroup,gene,cat,go
0,N0.HOG0002063,HBULBOSUM.FB19_011_3.PROJ.r1.chr6H_2G01211230.1,nat,[-]
1,N0.HOG0002063,HBULBOSUM.FB19_011_3.PROJ.r1.chr7H_2G01880360.1,nat,[-]
2,N0.HOG0002063,HBULBOSUM.FB19_011_3.PROJ.r1.chr3H_1G00518910.1,nat,[-]
3,N0.HOG0002063,HMURINUM.BCC2009.r1.5H_2G00340030.1,nat,[-]
4,N0.HOG0002063,HMUTICUM.GB90062.r1.7HG00774560.1,nat,[-]
...,...,...,...,...
953,N0.HOG0042274,HBULBOSUM.FB19_011_3.PROJ.r1.chr4H_2G00757210.1,nat,[-]
954,N0.HOG0042274,HBULBOSUM.FB19_011_3.PROJ.r1.chr4H_1G00671290.1,nat,[-]
955,N0.HOG0042274,HORVU.MOREX.PROJ.4HG00319200.1,nat,[-]
956,N0.HOG0042955,HGUSSONEANUM.BCC2005.r1.5HG00476740.1,nat,[-]


### Now that we have mapping per gene, we aggregate to get mapping per orthogroup

In [37]:
def aggregate_to_set(series):
    # Flatten lists and convert to set
    return set(item for sublist in series for item in sublist)

In [38]:
hgt_go = eggnog_hgt.groupby("orthogroup").agg({"gene": lambda x: set(x), "go": aggregate_to_set}).reset_index()
hgt_go.head()

,orthogroup,gene,go
0,N0.HOG0002063,"{HFLEXUOSUM.BCC2023.r1.2HG00198860.2, HMUTICUM...",{-}
1,N0.HOG0004282,"{HFLEXUOSUM.BCC2023.r1.5HG00613750.1, HEUCLAST...",{-}
2,N0.HOG0005093,"{HFLEXUOSUM.BCC2023.r1.ctg360G00870150.1, HFLE...","{GO:0032259, GO:0042445, GO:0010817, GO:004343..."
3,N0.HOG0007888,"{HCORDOBENSE.BCC2067.r1.6HG00722960.1, HEUCLAS...",{-}
4,N0.HOG0008649,{HERECTIFOLIUM.NGB6816.r1.4HG00529140.1},"{GO:0016020, GO:0009746, GO:0005886, GO:004222..."


In [39]:
nat_go = eggnog_nat.groupby("orthogroup").agg({"gene": lambda x: set(x), "go": aggregate_to_set}).reset_index()
nat_go.head()

,orthogroup,gene,go
0,N0.HOG0002063,{HBULBOSUM.FB19_011_3.PROJ.r1.chr6H_2G01211230...,{-}
1,N0.HOG0007888,"{HBOGDANII.H240.r1.7HG00797350.1, HMARINUM.BCC...",{-}
2,N0.HOG0008649,{HBULBOSUM.FB19_011_3.PROJ.r1.chr2H_2G00336600...,"{GO:0016020, GO:0009746, GO:0005886, GO:004222..."
3,N0.HOG0012002,"{HPUBIFLORUM.BCC2028.r1.7HG00928220.2, HJUBATU...","{GO:0016020, GO:0005886, GO:0055044, GO:003299..."
4,N0.HOG0013460,"{HMARINUM.H559.r1.4HG00278420.2, HCOMOSUM.NGB1...",{-}


In [40]:
cat_go = pd.merge(hgt_go, nat_go, on="orthogroup", suffixes=["_hgt", "_nat"], how="left")
cat_go

,orthogroup,gene_hgt,go_hgt,gene_nat,go_nat
0,N0.HOG0002063,"{HFLEXUOSUM.BCC2023.r1.2HG00198860.2, HMUTICUM...",{-},{HBULBOSUM.FB19_011_3.PROJ.r1.chr6H_2G01211230...,{-}
1,N0.HOG0004282,"{HFLEXUOSUM.BCC2023.r1.5HG00613750.1, HEUCLAST...",{-},NaN,NaN
2,N0.HOG0005093,"{HFLEXUOSUM.BCC2023.r1.ctg360G00870150.1, HFLE...","{GO:0032259, GO:0042445, GO:0010817, GO:004343...",NaN,NaN
3,N0.HOG0007888,"{HCORDOBENSE.BCC2067.r1.6HG00722960.1, HEUCLAS...",{-},"{HBOGDANII.H240.r1.7HG00797350.1, HMARINUM.BCC...",{-}
4,N0.HOG0008649,{HERECTIFOLIUM.NGB6816.r1.4HG00529140.1},"{GO:0016020, GO:0009746, GO:0005886, GO:004222...",{HBULBOSUM.FB19_011_3.PROJ.r1.chr2H_2G00336600...,"{GO:0016020, GO:0009746, GO:0005886, GO:004222..."
...,...,...,...,...,...
121,NUn.HOG0062451,{HFLEXUOSUM.BCC2023.r1.5HG00555880.1},"{GO:0016020, GO:0044437, GO:0008289, GO:000905...",NaN,NaN
122,NUn.HOG0066919,{HMUTICUM.GB90062.r1.5HG00510610.1},{-},NaN,NaN
123,NUn.HOG0067333,{HPATAGONICUM.BCC2065.r1.6HG00816250.2},"{GO:0005622, GO:0016020, GO:0012505, GO:190136...",NaN,NaN
124,NUn.HOG0067696,{HPUBIFLORUM.BCC2028.r1.5HG00648250.1},"{GO:0104004, GO:0009058, GO:0031326, GO:009030...",NaN,NaN


In [73]:
cat_go[(cat_go.go_hgt.notna()) & (cat_go.go_nat.notna())][["go_hgt", "go_nat"]].apply(lambda row: row["go_nat"] - row["go_hgt"], axis=1)

0                                                     {}
3                                                     {}
4                                                     {}
5                                                     {}
6                                                     {}
7                                                     {}
8                                                     {}
9                                                     {}
10                                                    {}
11                                                    {}
12                                                    {}
13                                                    {}
16                                                    {}
17                                                    {}
18                                                    {}
19                                                    {}
20                                                    {}
21                             

In [69]:
go_4 = cat_go.iloc[4,2]
go_4_nat = cat_go.iloc[4,4]

In [74]:
cat_go.iloc[39]

orthogroup                                        N0.HOG0034001
gene_hgt      {HCHILENSE.GRA1000.r1.6HG00748230.1, HMUTICUM....
go_hgt                                                      {-}
gene_nat      {HORVU.MOREX.PROJ.7HG00583010.1, HGUSSONEANUM....
go_nat        {GO:0050734, GO:0003824, GO:0003674, GO:001674...
Name: 39, dtype: object

### For enrichment analysis we need to unravel (explode) the sets and get one row for each orthogroup-GO term pair

In [75]:
exp_go_hgt = hgt_go.explode('go', ignore_index=True)

In [76]:
exp_go_hgt

,orthogroup,gene,go
0,N0.HOG0002063,"{HFLEXUOSUM.BCC2023.r1.2HG00198860.2, HMUTICUM...",-
1,N0.HOG0004282,"{HFLEXUOSUM.BCC2023.r1.5HG00613750.1, HEUCLAST...",-
2,N0.HOG0005093,"{HFLEXUOSUM.BCC2023.r1.ctg360G00870150.1, HFLE...",GO:0032259
3,N0.HOG0005093,"{HFLEXUOSUM.BCC2023.r1.ctg360G00870150.1, HFLE...",GO:0042445
4,N0.HOG0005093,"{HFLEXUOSUM.BCC2023.r1.ctg360G00870150.1, HFLE...",GO:0010817
...,...,...,...
2720,NUn.HOG0067696,{HPUBIFLORUM.BCC2028.r1.5HG00648250.1},GO:0006399
2721,NUn.HOG0067696,{HPUBIFLORUM.BCC2028.r1.5HG00648250.1},GO:0050896
2722,NUn.HOG0067696,{HPUBIFLORUM.BCC2028.r1.5HG00648250.1},GO:0034654
2723,NUn.HOG0067696,{HPUBIFLORUM.BCC2028.r1.5HG00648250.1},GO:0140110


In [77]:
exp_go_hgt[["go", "orthogroup"]].to_csv(ortho_dir / "hgt_genes_only_eggnog_exploded", sep='\t', index=False)